# Сопоставление сердечных масок MONAI Physio и TotalSegmentator v2

## Исследовательский вопрос

Проверено, меняет ли маршрут `SegmentChestTotalSegmentator` в MONAI Physio сердечные маски, которые проект получает при отдельном запуске TotalSegmentator v2 (TSv2), и можно ли использовать его как техническую альтернативу. Обёртка вызывает задачи TotalSegmentator и объединяет их ответы. Сравнение проверяет подготовку КТ, область обработки, перенос на исходную сетку и объединение меток; оно не сопоставляет два независимых способа определения истинной границы сердца.

**Статус результата.** Полный маршрут выполнен на фазе 00 Ника, затем на всех 12 фазах его 4D-КТ и на всех 20 фазах исторической контрастной КТ. Одноимённые автоматические маски близки, но устойчивого преимущества MONAI Physio по анатомической точности не показано. Проектный TSv2 остаётся основным источником масок. Это решение относится к проверенным сериям и настройкам.

## Данные, классы и метрики

Фаза 00 Ника имеет сетку `512×512×256` с шагом `0,765625×0,765625×0,625 мм`. Полный маршрут выполнил задачи `total`, `heartchambers_highres`, `tissue_4_types`, `lung_vessels` и `body`. Сравнимый объект — **объединение пяти классов** `heartchambers_highres`: миокарда и четырёх камер. В общей карте MONAI это метки `140–144`; проектный выход состоит из пяти отдельных масок той же задачи.

Метка `51` общей карты — остаток класса `total/heart` после наложения других классов. Сырой `total/heart`, остаточная метка и объединение миокарда с камерами имеют разную семантику. Их пересечение не служит оценкой точности. Размер, аффинная матрица и ориентация сравниваемых масок сверены на одной физической сетке. Заголовок NIfTI сообщает единицу `unknown`; миллиметры приняты по происхождению КТ как **явное допущение**, от которого зависят объёмы в миллилитрах.

Для бинарных масок A и B коэффициент Dice измеряет перекрытие: `2|A∩B|/(|A|+|B|)`. Симметрическая разность — объём вокселей, принадлежащих ровно одной маске. Здесь обе метрики описывают **согласие автоматических выходов**. Для оценки анатомической ошибки нужен отдельный принятый контур соответствующей фазы.

## Фаза 00 Ника: полный и частичный маршруты

Пять задач полного маршрута завершились; общая карта и отдельные маски сохранены локально в исключённом из Git `runs/`. Для пяти одноимённых классов MONAI и проектного TSv2 совпали форма сетки и аффинная матрица. Объём объединения MONAI — **569,387 мл**, проектного объединения — **571,987 мл**; Dice **0,990910**, коэффициент Жаккара (доля пересечения в объединении масок) **0,981983**, симметрическая разность **10,375 мл**. Различия предобработки и области обработки остаются возможными причинами несовпадения. Связь прежнего проектного выхода с фазой установлена манифестом, но прямой хэш исходной КТ для того запуска не найден.

Диагностический **частичный** режим исполнил `total` и `heartchambers_highres`, а ответы остальных трёх задач заменил нулями; его манифест помечен `full_monai_physio_workflow=false`. Объём объединения — **572,704 мл**; относительно проектного объединения Dice **0,994125**, симметрическая разность **6,725 мл**. Между частичным и полным объединениями различались **10 793 вокселя**, Dice между ними **0,996538**. Вспомогательные задачи и наложение меток изменяют даже классы `140–144`. Более высокий Dice частичного режима не означает лучшую анатомическую сегментацию.

Полное объединение выходит за пределы сырого `total/heart` на **16,570 мл**, проектное — на **15,738 мл**. Эти величины характеризуют пространственное отношение **разных классов**. Подробные данные приведены в [отчёте пилота](mp01_02_отчёт_пилота.md).

При первом запуске ITK не открыл NIfTI по Windows-пути с Unicode-символами; тот же файл прочитан через временный ASCII-алиас. Для двух наборов весов потребовалось отдельное разрешённое получение файлов. После этого полный маршрут завершился. Это условия воспроизведения, а не свидетельство качества масок; КТ в сеть при загрузке весов не передавалась.

## Старая КТ и полный цикл Ника

Историческая контрастная КТ содержит **20 фаз** на сетке `512×512×103` с шагом `0,473×0,473×2,5 мм`. На всех фазах полный маршрут MONAI Physio сопоставлен с проектным TSv2 для тех же пяти классов. Средний Dice по фазам составил **0,9748** для ПЖ и **0,9880** для ЛЖ; наибольшая разница абсолютного объёма ПЖ — **10,41 мл**. ПЖ наиболее чувствителен к смене маршрута. Прямые кандидатные контуры для фаз 0% и 40% созданы после просмотра TSv2; они не являются ослеплённым независимым референсом. Для миокарда прямого контура нет. Числа по каждому классу и фазе приведены в [отчёте по старой КТ](mp01_old_ct_report.md).

На современной КТ Ника маршрут исполнен для **всех 12 фаз**. Средний Dice относительно проектного TSv2 составляет **0,9815–0,9919** по пяти классам. Наибольшая разница абсолютного объёма камеры — **6,86 мл** для ПЖ; наибольшая разница изменения объёма относительно фазы 00 — **3,30 мл** для ПЖ. Сравниваются два автоматических выхода одной семьи моделей. Высокая близость не устанавливает точность стенок, клапанных границ или межфазных изменений. [Пофазный отчёт Ника](mp01_nix_full_cycle_report.md) и агрегаты сохраняют метрики и контрольные суммы.

**Решение:** MONAI Physio пригоден как прослеживаемый альтернативный маршрут получения кандидатных масок и для анализа чувствительности последующих расчётов. Оснований менять основной TSv2 ради повышения анатомической точности нет. Маски остаются автоматическими исследовательскими входами, в том числе для регистрации и FEM.

## Отдельная предобученная модель NVIDIA

Модель `NV-Segment-CTMR` через MONAI Physio проверялась **отдельно от обёртки TSv2** на фазах 0% и 40% исторической КТ без обучения на данных проекта. Стандартный режим `CT_BODY` выдал сердце целиком, но не выдал четыре камеры и миокард; явные запросы пяти классов технически сработали. Изменения их объёма заметно расходятся с TSv2 и прямыми кандидатными контурами, особенно для ЛЖ, ПЖ и ПП. В этой серии модель **не принята как замена TSv2 для количественной динамики камер**.

Стандартная наружная маска сердца имела Dice **0,948** и **0,939** относительно одноимённой маски TSv2 на двух фазах. Независимого принятого наружного контура нет; эти числа не показывают, какая маска анатомически вернее. Подробные результаты и ограничения выборки — в [отдельном отчёте](mp01_nvsegment_old_report.md).

In [ ]:
import os, subprocess, sys
# Optional local repeat; model download access is controlled by the launcher flags.
# The launcher enforces a fresh child directory inside ignored runs/.
if os.environ.get('MP01_RUN_MONAI') == '1':
    command = [sys.executable, 'mp01_02_run_monai.py', '--ct', os.environ['MP01_CT'], '--output-dir', os.environ.get('MP01_MONAI_OUTPUT', 'runs/mp01_02_monai_candidate')]
    if os.environ.get('MP01_CONTRAST') == '1': command.append('--contrast')
    if os.environ.get('MP01_HEART_ONLY_DIAGNOSTIC') == '1': command.append('--heart-only-diagnostic')
    subprocess.run(command, check=True)


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
required = ('MP01_CT', 'MP01_BASELINE_DIR', 'MP01_CANDIDATE_DIR')
missing = [key for key in required if not os.environ.get(key)]
if missing:
    print('Local inputs not configured:', ', '.join(missing))
else:
    script = Path('mp01_02_compare_heart.py')
    output = Path('runs') / f"mp01_02_heart_comparison_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}.json"
    command = [sys.executable, str(script), '--ct', os.environ['MP01_CT'], '--baseline-dir', os.environ['MP01_BASELINE_DIR'], '--candidate-dir', os.environ['MP01_CANDIDATE_DIR'], '--output', str(output)]
    if os.environ.get('MP01_WHOLE_HEART'): command.extend(['--whole-heart', os.environ['MP01_WHOLE_HEART']])
    if os.environ.get('MP01_VERIFIED_MM') == '1': command.append('--assume-mm')
    subprocess.run(command, check=True)


## Контроль качества и граница применения

Локальные отчёты в `runs/` содержат входные хэши, параметры и аффинные матрицы. При повторном сравнении требуется сверить фазу, версию и веса TSv2, задачу, режим `fast`, область обработки, ориентацию RAS, шаг, аффинную матрицу и интерполяцию. Имя файла этих свойств не подтверждает. Принятие миллиметров при `xyzt_units=unknown` требует проверки происхождения DICOM/NIfTI; код разрешает его только через явный флаг `MP01_VERIFIED_MM=1`.

Проверка устанавливает **внутреннее межмаршрутное согласие** и включает ограниченный просмотр контуров. Независимой ручной анатомической проверки современных фаз Ника нет. Ошибки сегментации, различия дыхания и преобразования между программами остаются разными источниками неопределённости. Следующие этапы используют маски с этим статусом для контроля поверхности ([20.19](20.19_MP02_Маска_поверхность_FEM.ipynb)) и регистрации фаз ([21.15](21.15_MP03_Регистрация_фаз_4D_КТ.ipynb)). Передача маски в расчёт не повышает доказательный статус её границы.